In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import torch
from sam2.build_sam import build_sam2_video_predictor


# -----------------------------
# CONFIG
# -----------------------------
VIDEO_PATH = r"VideosAnalisis\clip 3 ‐ Hecho con Clipchamp.mp4"
MAP_PATH   = r"beachvolleyballcourt.png"

SAM2_CKPT = os.path.join("checkpoints", "sam2.1_hiera_tiny.pt")
SAM2_CFG  = r"configs/sam2.1/sam2.1_hiera_t.yaml"

FRAMES_DIR = "./_sam2_frames"
OUT_DIR    = "./outputs"
OUT_CSV    = os.path.join(OUT_DIR, "ball_sam2_track_h2d.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROI_WINDOW_NAME   = "Selecciona la pelota (ROI)"
ROI_PREVIEW_MAX_W = 960
ROI_PREVIEW_MAX_H = 540
INIT_FRAME_GUESS  = 10
NONBLACK_SEARCH_LIMIT = 200

# Homografía
N_POINTS = 6  # >=4

# Visualización
TARGET_VIDEO_WIDTH = 900
MINIMAP_HEIGHT_RATIO = 0.55

# Mezcla adaptativa (aplanado)
# air_score ~= (yg - y) / diam_px  -> alto si el centroide está “muy arriba” respecto al borde inferior
AIR_T0 = 0.20   # umbral bajo
AIR_T1 = 0.55   # umbral alto (>= => casi todo suelo)
SMOOTH_ALPHA_MAP = 0.35  # suavizado en minimapa (0..1)


# -----------------------------
# Helpers generales
# -----------------------------
def sanitize_folder_name(name):
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    name = re.sub(r'[^\w\s\-.]', '_', name)
    name = re.sub(r'\s+', '_', name)
    return name.strip('_')

def ensure_dirs():
    os.makedirs(FRAMES_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)

def extract_frames(video_path, frames_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[INFO] Extrayendo {frame_count} frames...")
    for i in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            frame_count = i
            break
        cv2.imwrite(os.path.join(frames_path, f"{i:06d}.jpg"), frame)
        if i % 100 == 0:
            print(f"[INFO] Extraídos {i}/{frame_count} frames...")
    cap.release()
    print(f"[INFO] Extracción completada: {frame_count} frames guardados en {frames_path}")
    return fps, W, H, frame_count

def read_frame_from_video(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

def is_black_frame(frame, thr_mean=8.0):
    if frame is None:
        return True
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(gray.mean()) < thr_mean

def pick_nonblack_frame_idx(video_path, start_idx, frame_count, search_limit=200):
    start_idx = max(0, min(int(start_idx), max(0, frame_count - 1)))
    max_idx = min(frame_count - 1, start_idx + int(search_limit))
    for idx in range(start_idx, max_idx + 1):
        fr = read_frame_from_video(video_path, idx)
        if fr is not None and not is_black_frame(fr):
            return idx, fr
    return start_idx, read_frame_from_video(video_path, start_idx)

def load_or_create_frame_jpg(frames_path, video_path, frame_idx):
    frame_file = os.path.join(frames_path, f"{int(frame_idx):06d}.jpg")
    if os.path.exists(frame_file):
        fr = cv2.imread(frame_file)
        if fr is not None:
            return fr
    fr = read_frame_from_video(video_path, frame_idx)
    if fr is None:
        return None
    cv2.imwrite(frame_file, fr)
    return fr

def resize_for_preview(frame, max_w=960, max_h=540):
    H, W = frame.shape[:2]
    scale = min(max_w / W, max_h / H, 1.0)
    new_w = int(round(W * scale))
    new_h = int(round(H * scale))
    preview = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA) if scale < 1.0 else frame.copy()
    sx = W / new_w
    sy = H / new_h
    return preview, sx, sy

def select_roi_small_window(frame_bgr):
    preview, sx, sy = resize_for_preview(frame_bgr, ROI_PREVIEW_MAX_W, ROI_PREVIEW_MAX_H)
    cv2.namedWindow(ROI_WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(ROI_WINDOW_NAME, preview.shape[1], preview.shape[0])
    print("[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)")
    roi = cv2.selectROI(ROI_WINDOW_NAME, preview, fromCenter=False, showCrosshair=True)
    cv2.destroyWindow(ROI_WINDOW_NAME)

    x, y, w, h = roi
    if w == 0 or h == 0:
        raise RuntimeError("ROI vacío. Vuelve a ejecutar y selecciona una caja válida.")

    x0 = float(x) * sx
    y0 = float(y) * sy
    x1 = float(x + w) * sx
    y1 = float(y + h) * sy
    return np.array([x0, y0, x1, y1], dtype=np.float32)


# -----------------------------
# Máscara -> puntos
# -----------------------------
def mask_points(binmask: np.ndarray):
    ys, xs = np.where(binmask > 0)
    if len(xs) == 0:
        return (np.nan, np.nan, np.nan, np.nan, 0, 0.0)

    # centroide
    cx = float(xs.mean())
    cy = float(ys.mean())

    # "suelo" (punto inferior)
    y_max = int(ys.max())
    xs_bottom = xs[ys == y_max]
    xg = float(xs_bottom.mean()) if len(xs_bottom) else cx
    yg = float(y_max)

    # bbox aproximado para diametro
    x_min, x_max = int(xs.min()), int(xs.max())
    y_min, y_max2 = int(ys.min()), int(ys.max())
    bw = max(1, x_max - x_min + 1)
    bh = max(1, y_max2 - y_min + 1)
    diam = float(max(bw, bh))  # diámetro px aprox

    area = int(len(xs))
    return (cx, cy, xg, yg, area, diam)


# -----------------------------
# UI homografía (clics)
# -----------------------------
def get_screen_size():
    screen = np.zeros((1, 1, 3), dtype=np.uint8)
    cv2.namedWindow("_tmp", cv2.WINDOW_NORMAL)
    cv2.imshow("_tmp", screen)
    cv2.waitKey(1)
    _, _, w, h = cv2.getWindowImageRect("_tmp")
    cv2.destroyWindow("_tmp")
    return w, h

def resize_to_fit(img, max_w, max_h):
    h, w = img.shape[:2]
    scale = min(max_w / w, max_h / h, 1.0)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    if scale < 1.0:
        out = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    else:
        out = img.copy()
    return out, (w / new_w), (h / new_h)

def _mouse_cb_collect_points(event, x, y, flags, params):
    if event != cv2.EVENT_LBUTTONDOWN:
        return
    pts = params["points"]
    if len(pts) >= params["max_points"]:
        return
    img = params["image"]
    wname = params["wname"]

    pts.append([x, y])
    cv2.circle(img, (x, y), 7, (0, 0, 255), -1)
    cv2.putText(img, f"{len(pts)}", (x + 8, y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.imshow(wname, img)

    if len(pts) == params["max_points"]:
        cv2.waitKey(250)
        cv2.destroyWindow(wname)

def collect_correspondences(imgA, imgB, n_points=6, titleA="VIDEO", titleB="MAPA"):
    sw, sh = get_screen_size()
    max_w = int(sw * 0.85)
    max_h = int(sh * 0.85)

    # A
    dispA, sxA, syA = resize_to_fit(imgA, max_w, max_h)
    ptsA_disp = []
    imgA_draw = dispA.copy()
    wA = f"{titleA}: clica {n_points} puntos (ESC cancela)"
    cv2.namedWindow(wA, cv2.WINDOW_NORMAL)
    cv2.imshow(wA, imgA_draw)
    cv2.setMouseCallback(wA, _mouse_cb_collect_points,
                         {"points": ptsA_disp, "image": imgA_draw, "wname": wA, "max_points": n_points})
    while True:
        k = cv2.waitKey(20) & 0xFF
        if k == 27:
            cv2.destroyAllWindows()
            raise RuntimeError("Cancelado puntos VIDEO.")
        if len(ptsA_disp) >= n_points:
            break
    ptsA = np.array([[p[0] * sxA, p[1] * syA] for p in ptsA_disp], dtype=np.float32)

    # B
    dispB, sxB, syB = resize_to_fit(imgB, max_w, max_h)
    ptsB_disp = []
    imgB_draw = dispB.copy()
    wB = f"{titleB}: clica {n_points} puntos correspondientes (ESC cancela)"
    cv2.namedWindow(wB, cv2.WINDOW_NORMAL)
    cv2.imshow(wB, imgB_draw)
    cv2.setMouseCallback(wB, _mouse_cb_collect_points,
                         {"points": ptsB_disp, "image": imgB_draw, "wname": wB, "max_points": n_points})
    while True:
        k = cv2.waitKey(20) & 0xFF
        if k == 27:
            cv2.destroyAllWindows()
            raise RuntimeError("Cancelado puntos MAPA.")
        if len(ptsB_disp) >= n_points:
            break
    ptsB = np.array([[p[0] * sxB, p[1] * syB] for p in ptsB_disp], dtype=np.float32)

    cv2.destroyAllWindows()
    return ptsA, ptsB


# -----------------------------
# Mezcla adaptativa en minimapa
# -----------------------------
def clamp01(x):
    return float(max(0.0, min(1.0, x)))

def mix2(a, b, w):
    # w=0 => a ; w=1 => b
    return (1.0 - w) * a + w * b

def air_weight(cy, yg, diam):
    """
    Estima cuánto está "en aire" usando separación vertical relativa al tamaño.
    air_score = (yg - cy) / diam
    -> si centroide está bastante por encima del borde inferior, pelota probablemente alta.
    Devuelve w in [0,1] (0 usar centroide; 1 usar suelo)
    """
    if not np.isfinite(cy) or not np.isfinite(yg) or not np.isfinite(diam) or diam <= 1e-6:
        return 0.0
    score = (float(yg) - float(cy)) / float(diam)
    # mapear score [AIR_T0..AIR_T1] -> w [0..1]
    w = (score - AIR_T0) / (AIR_T1 - AIR_T0 + 1e-9)
    return clamp01(w)


# -----------------------------
# MAIN
# -----------------------------
def main():
    ensure_dirs()

    if not os.path.exists(SAM2_CKPT):
        raise FileNotFoundError(
            f"No existe el checkpoint: {SAM2_CKPT}\n"
            f"Ejecuta antes: python scripts/download_sam2_ckpt.py"
        )
    if not os.path.exists(MAP_PATH):
        raise FileNotFoundError(f"No existe el mapa: {MAP_PATH}")

    # Frames folder
    video_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    video_stem_clean = sanitize_folder_name(video_stem)
    frames_path = os.path.join(FRAMES_DIR, video_stem_clean)
    os.makedirs(frames_path, exist_ok=True)

    # Extraer frames si no existen
    existing_jpgs = [f for f in os.listdir(frames_path) if f.lower().endswith(".jpg")]
    if len(existing_jpgs) == 0:
        fps, W, H, frame_count = extract_frames(VIDEO_PATH, frames_path)
    else:
        cap0 = cv2.VideoCapture(VIDEO_PATH)
        if not cap0.isOpened():
            raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")
        fps = cap0.get(cv2.CAP_PROP_FPS)
        W = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_count = int(cap0.get(cv2.CAP_PROP_FRAME_COUNT))
        cap0.release()

    # Frame inicial no negro
    init_frame_idx, _ = pick_nonblack_frame_idx(
        VIDEO_PATH, start_idx=INIT_FRAME_GUESS, frame_count=frame_count, search_limit=NONBLACK_SEARCH_LIMIT
    )
    init_frame = load_or_create_frame_jpg(frames_path, VIDEO_PATH, init_frame_idx)
    if init_frame is None:
        raise RuntimeError("No pude cargar frame inicial para ROI.")
    print(f"[INFO] Usando frame inicial: {init_frame_idx:06d}.jpg")

    # ROI
    box = select_roi_small_window(init_frame)
    print(f"[INFO] ROI seleccionado: {box}")

    # SAM2 predictor
    print(f"[INFO] Cargando SAM2 en {DEVICE} ...")
    predictor = build_sam2_video_predictor(SAM2_CFG, SAM2_CKPT, device=DEVICE)

    print(f"[INFO] Inicializando estado de SAM2 con carpeta: {frames_path}")
    with torch.inference_mode():
        try:
            state = predictor.init_state(video_path=frames_path)
        except TypeError:
            state = predictor.init_state(frames_path)

    OBJ_ID = 1
    with torch.inference_mode():
        try:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), obj_id=OBJ_ID, box=box)
        except TypeError:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), object_id=OBJ_ID, box=box)

    # Propagar
    print("[INFO] Propagando máscara en el vídeo...")
    rows = []
    with torch.inference_mode():
        for f_idx, obj_ids, masks in predictor.propagate_in_video(state):
            f_idx = int(f_idx)
            obj_ids_list = [int(o) for o in obj_ids]
            if OBJ_ID not in obj_ids_list:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "xg": np.nan, "yg": np.nan, "area": 0, "diam": np.nan})
                continue

            j = obj_ids_list.index(OBJ_ID)
            mask = masks[j]

            mask_np = mask.squeeze().detach().float().cpu().numpy()
            binmask = (mask_np > 0).astype(np.uint8)

            cx, cy, xg, yg, area, diam = mask_points(binmask)
            if area < 5:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "xg": np.nan, "yg": np.nan, "area": area, "diam": diam})
                continue

            rows.append({"frame": f_idx, "x": cx, "y": cy, "xg": xg, "yg": yg, "area": area, "diam": diam})

            if f_idx % 50 == 0:
                print(f"[INFO] Procesado frame {f_idx}/{frame_count}")

    df = pd.DataFrame(rows).sort_values("frame")
    df.to_csv(OUT_CSV, index=False)
    print(f"[OK] CSV guardado: {OUT_CSV} (incluye xg,yg,diam)")

    # Homografía (clics)
    mapa = cv2.imread(MAP_PATH)
    if mapa is None:
        raise RuntimeError(f"No se pudo leer mapa: {MAP_PATH}")

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir video para visualizar: {VIDEO_PATH}")
    ret, first_frame = cap.read()
    if not ret:
        cap.release()
        raise RuntimeError("No pude leer primer frame para homografía.")
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    print("[INFO] Selecciona puntos para homografía VIDEO -> MAPA")
    if N_POINTS < 4:
        raise ValueError("N_POINTS debe ser >= 4.")
    pA, pB = collect_correspondences(first_frame, mapa, n_points=N_POINTS, titleA="VIDEO", titleB="MAPA")
    Hm, _ = cv2.findHomography(pA, pB, method=cv2.RANSAC)
    if Hm is None:
        cap.release()
        raise RuntimeError("No se pudo calcular homografía.")

    # Reproducir con minimapa (mezcla adaptativa)
    delay = max(1, int(round(1000 / fps))) if fps and fps > 1e-3 else 30
    last_map = None  # suavizado (xm,ym)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        f_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
        mapa_out = mapa.copy()

        row_df = df[df["frame"] == f_idx]
        if len(row_df) > 0:
            r = row_df.iloc[0]
            if pd.notna(r["x"]) and pd.notna(r["y"]) and pd.notna(r["xg"]) and pd.notna(r["yg"]):
                cx, cy = float(r["x"]), float(r["y"])
                xg, yg = float(r["xg"]), float(r["yg"])
                diam = float(r["diam"]) if pd.notna(r["diam"]) else 1.0

                # proyectar ambos puntos a mapa
                pt_c = np.array([[cx, cy]], dtype=np.float32).reshape(-1, 1, 2)
                pt_g = np.array([[xg, yg]], dtype=np.float32).reshape(-1, 1, 2)
                xm_c, ym_c = cv2.perspectiveTransform(pt_c, Hm).reshape(-1, 2)[0]
                xm_g, ym_g = cv2.perspectiveTransform(pt_g, Hm).reshape(-1, 2)[0]

                # peso según "en aire"
                w = air_weight(cy, yg, diam)  # 0..1
                xm = mix2(xm_c, xm_g, w)
                ym = mix2(ym_c, ym_g, w)

                # suavizado en minimapa
                if SMOOTH_ALPHA_MAP > 0:
                    if last_map is None:
                        last_map = (xm, ym)
                    else:
                        xm = SMOOTH_ALPHA_MAP * xm + (1 - SMOOTH_ALPHA_MAP) * last_map[0]
                        ym = SMOOTH_ALPHA_MAP * ym + (1 - SMOOTH_ALPHA_MAP) * last_map[1]
                        last_map = (xm, ym)

                # dibujar en frame (debug: centroide amarillo, ground cian)
                cv2.circle(frame, (int(cx), int(cy)), 6, (0, 255, 255), -1)
                cv2.circle(frame, (int(xg), int(yg)), 6, (255, 255, 0), -1)
                cv2.putText(frame, f"BALL w={w:.2f}", (int(cx) + 10, int(cy) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                # dibujar en minimapa
                cv2.circle(mapa_out, (int(xm), int(ym)), 18, (255, 255, 255), -1)
                cv2.putText(mapa_out, "Pelota", (int(xm) + 10, int(ym) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        # combinar
        fh, fw = frame.shape[:2]
        frame_r = cv2.resize(frame, (TARGET_VIDEO_WIDTH, int(fh * TARGET_VIDEO_WIDTH / fw)))
        target_h = frame_r.shape[0]

        minimap_target_h = int(target_h * MINIMAP_HEIGHT_RATIO)
        mapa_r = cv2.resize(mapa_out, (int(mapa_out.shape[1] * minimap_target_h / mapa_out.shape[0]), minimap_target_h))

        padding_top = (target_h - minimap_target_h) // 2
        padding_bottom = target_h - minimap_target_h - padding_top
        mapa_padded = cv2.copyMakeBorder(mapa_r, padding_top, padding_bottom, 0, 0,
                                         cv2.BORDER_CONSTANT, value=[0, 0, 0])

        combined = np.hstack((frame_r, mapa_padded))
        cv2.imshow("Video + Minimapa", combined)

        if cv2.waitKey(delay) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    print("[OK] Fin.")


if __name__ == "__main__":
    main()


[INFO] Usando frame inicial: 000010.jpg
[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)
[INFO] ROI seleccionado: [558. 552. 588. 578.]
[INFO] Cargando SAM2 en cpu ...
[INFO] Inicializando estado de SAM2 con carpeta: ./_sam2_frames\clip_3___Hecho_con_Clipchamp


frame loading (JPEG): 100%|██████████| 448/448 [01:44<00:00,  4.30it/s]
c:\Users\wailb\anaconda3\envs\Beachvolley-Homography\Lib\site-packages\sam2\sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\wailb\anaconda3\envs\Beachvolley-Homography\Lib\site-packages\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


[INFO] Propagando máscara en el vídeo...


propagate in video:   9%|▉         | 41/438 [23:20<3:04:07, 27.83s/it]

[INFO] Procesado frame 50/448


...

[INFO] Procesado frame 350/448


propagate in video:  89%|████████▉ | 391/438 [1:31:19<18:27, 23.56s/it]

[INFO] Procesado frame 400/448


propagate in video: 100%|██████████| 438/438 [1:42:16<00:00, 14.01s/it]


[OK] CSV guardado: ./outputs\ball_sam2_track_h2d.csv (incluye xg,yg,diam)
[INFO] Selecciona puntos para homografía VIDEO -> MAPA
[OK] Fin.


In [8]:
# ============================================================
# CHUNK 1/2 — INFERENCIA SAM2 + CSV (guarda x,y,xg,yg,area,diam)
# ------------------------------------------------------------
# Ejecuta SAM2 una vez y genera:
#   ./outputs/ball_sam2_track_h2d.csv
#
# Luego podrás usar el CHUNK 2 para homografía+replay SIN reinferir.
# ============================================================

import os
import re
import cv2
import numpy as np
import pandas as pd
import torch
from sam2.build_sam import build_sam2_video_predictor

# -----------------------------
# CONFIG
# -----------------------------
VIDEO_PATH = r"VideosAnalisis\clip 3 ‐ Hecho con Clipchamp.mp4"
SAM2_CKPT = os.path.join("checkpoints", "sam2.1_hiera_tiny.pt")
SAM2_CFG  = r"configs/sam2.1/sam2.1_hiera_t.yaml"

FRAMES_DIR = "./_sam2_frames"
OUT_DIR    = "./outputs"
OUT_CSV    = os.path.join(OUT_DIR, "ball_sam2_track_h2d.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROI_WINDOW_NAME   = "Selecciona la pelota (ROI)"
ROI_PREVIEW_MAX_W = 960
ROI_PREVIEW_MAX_H = 540
INIT_FRAME_GUESS  = 10
NONBLACK_SEARCH_LIMIT = 200

# -----------------------------
# Helpers
# -----------------------------
def sanitize_folder_name(name):
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    name = re.sub(r'[^\w\s\-.]', '_', name)
    name = re.sub(r'\s+', '_', name)
    return name.strip('_')

def ensure_dirs():
    os.makedirs(FRAMES_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)

def extract_frames(video_path, frames_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[INFO] Extrayendo {frame_count} frames...")
    for i in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            frame_count = i
            break
        cv2.imwrite(os.path.join(frames_path, f"{i:06d}.jpg"), frame)
        if i % 100 == 0:
            print(f"[INFO] Extraídos {i}/{frame_count} frames...")
    cap.release()
    print(f"[INFO] Extracción completada: {frame_count} frames guardados en {frames_path}")
    return fps, W, H, frame_count

def read_frame_from_video(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

def is_black_frame(frame, thr_mean=8.0):
    if frame is None:
        return True
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(gray.mean()) < thr_mean

def pick_nonblack_frame_idx(video_path, start_idx, frame_count, search_limit=200):
    start_idx = max(0, min(int(start_idx), max(0, frame_count - 1)))
    max_idx = min(frame_count - 1, start_idx + int(search_limit))
    for idx in range(start_idx, max_idx + 1):
        fr = read_frame_from_video(video_path, idx)
        if fr is not None and not is_black_frame(fr):
            return idx, fr
    return start_idx, read_frame_from_video(video_path, start_idx)

def load_or_create_frame_jpg(frames_path, video_path, frame_idx):
    frame_file = os.path.join(frames_path, f"{int(frame_idx):06d}.jpg")
    if os.path.exists(frame_file):
        fr = cv2.imread(frame_file)
        if fr is not None:
            return fr
    fr = read_frame_from_video(video_path, frame_idx)
    if fr is None:
        return None
    cv2.imwrite(frame_file, fr)
    return fr

def resize_for_preview(frame, max_w=960, max_h=540):
    H, W = frame.shape[:2]
    scale = min(max_w / W, max_h / H, 1.0)
    new_w = int(round(W * scale))
    new_h = int(round(H * scale))
    preview = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA) if scale < 1.0 else frame.copy()
    sx = W / new_w
    sy = H / new_h
    return preview, sx, sy

def select_roi_small_window(frame_bgr):
    preview, sx, sy = resize_for_preview(frame_bgr, ROI_PREVIEW_MAX_W, ROI_PREVIEW_MAX_H)
    cv2.namedWindow(ROI_WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(ROI_WINDOW_NAME, preview.shape[1], preview.shape[0])
    print("[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)")
    roi = cv2.selectROI(ROI_WINDOW_NAME, preview, fromCenter=False, showCrosshair=True)
    cv2.destroyWindow(ROI_WINDOW_NAME)

    x, y, w, h = roi
    if w == 0 or h == 0:
        raise RuntimeError("ROI vacío. Vuelve a ejecutar y selecciona una caja válida.")

    x0 = float(x) * sx
    y0 = float(y) * sy
    x1 = float(x + w) * sx
    y1 = float(y + h) * sy
    return np.array([x0, y0, x1, y1], dtype=np.float32)

def mask_points(binmask: np.ndarray):
    ys, xs = np.where(binmask > 0)
    if len(xs) == 0:
        return (np.nan, np.nan, np.nan, np.nan, 0, np.nan)

    cx = float(xs.mean())
    cy = float(ys.mean())

    y_max = int(ys.max())
    xs_bottom = xs[ys == y_max]
    xg = float(xs_bottom.mean()) if len(xs_bottom) else cx
    yg = float(y_max)

    x_min, x_max = int(xs.min()), int(xs.max())
    y_min, y_max2 = int(ys.min()), int(ys.max())
    bw = max(1, x_max - x_min + 1)
    bh = max(1, y_max2 - y_min + 1)
    diam = float(max(bw, bh))

    area = int(len(xs))
    return (cx, cy, xg, yg, area, diam)

# -----------------------------
# MAIN
# -----------------------------
def main():
    ensure_dirs()

    if not os.path.exists(SAM2_CKPT):
        raise FileNotFoundError(
            f"No existe el checkpoint: {SAM2_CKPT}\n"
            f"Ejecuta antes: python scripts/download_sam2_ckpt.py"
        )

    video_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    video_stem_clean = sanitize_folder_name(video_stem)
    frames_path = os.path.join(FRAMES_DIR, video_stem_clean)
    os.makedirs(frames_path, exist_ok=True)

    existing_jpgs = [f for f in os.listdir(frames_path) if f.lower().endswith(".jpg")]
    if len(existing_jpgs) == 0:
        fps, W, H, frame_count = extract_frames(VIDEO_PATH, frames_path)
    else:
        cap0 = cv2.VideoCapture(VIDEO_PATH)
        if not cap0.isOpened():
            raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")
        fps = cap0.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap0.get(cv2.CAP_PROP_FRAME_COUNT))
        cap0.release()
        print(f"[INFO] Usando frames existentes ({len(existing_jpgs)} jpg). frame_count={frame_count}")

    init_frame_idx, _ = pick_nonblack_frame_idx(
        VIDEO_PATH, start_idx=INIT_FRAME_GUESS, frame_count=frame_count, search_limit=NONBLACK_SEARCH_LIMIT
    )
    init_frame = load_or_create_frame_jpg(frames_path, VIDEO_PATH, init_frame_idx)
    if init_frame is None:
        raise RuntimeError("No pude cargar frame inicial para ROI.")
    print(f"[INFO] Usando frame inicial: {init_frame_idx:06d}.jpg")

    box = select_roi_small_window(init_frame)
    print(f"[INFO] ROI seleccionado: {box}")

    print(f"[INFO] Cargando SAM2 en {DEVICE} ...")
    predictor = build_sam2_video_predictor(SAM2_CFG, SAM2_CKPT, device=DEVICE)

    print(f"[INFO] Inicializando estado de SAM2 con carpeta: {frames_path}")
    with torch.inference_mode():
        try:
            state = predictor.init_state(video_path=frames_path)
        except TypeError:
            state = predictor.init_state(frames_path)

    OBJ_ID = 1
    with torch.inference_mode():
        try:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), obj_id=OBJ_ID, box=box)
        except TypeError:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), object_id=OBJ_ID, box=box)

    print("[INFO] Propagando máscara en el vídeo...")
    rows = []
    with torch.inference_mode():
        for f_idx, obj_ids, masks in predictor.propagate_in_video(state):
            f_idx = int(f_idx)
            obj_ids_list = [int(o) for o in obj_ids]
            if OBJ_ID not in obj_ids_list:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "xg": np.nan, "yg": np.nan, "area": 0, "diam": np.nan})
                continue

            j = obj_ids_list.index(OBJ_ID)
            mask = masks[j]
            mask_np = mask.squeeze().detach().float().cpu().numpy()
            binmask = (mask_np > 0).astype(np.uint8)

            cx, cy, xg, yg, area, diam = mask_points(binmask)
            if area < 5:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "xg": np.nan, "yg": np.nan, "area": area, "diam": diam})
                continue

            rows.append({"frame": f_idx, "x": cx, "y": cy, "xg": xg, "yg": yg, "area": area, "diam": diam})

            if f_idx % 50 == 0:
                print(f"[INFO] Procesado frame {f_idx}/{frame_count}")

    df = pd.DataFrame(rows).sort_values("frame")
    df.to_csv(OUT_CSV, index=False)
    print(f"[OK] CSV guardado: {OUT_CSV}")
    print("[OK] Columnas:", list(df.columns))

if __name__ == "__main__":
    main()


[INFO] Usando frames existentes (448 jpg). frame_count=442
[INFO] Usando frame inicial: 000010.jpg
[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)
[INFO] ROI seleccionado: [562. 554. 588. 578.]
[INFO] Cargando SAM2 en cpu ...
[INFO] Inicializando estado de SAM2 con carpeta: ./_sam2_frames\clip_3___Hecho_con_Clipchamp


frame loading (JPEG): 100%|██████████| 448/448 [00:48<00:00,  9.22it/s]
c:\Users\wailb\anaconda3\envs\Beachvolley-Homography\Lib\site-packages\sam2\sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\wailb\anaconda3\envs\Beachvolley-Homography\Lib\site-packages\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


[INFO] Propagando máscara en el vídeo...


propagate in video:   9%|▉         | 41/438 [05:47<29:12,  4.41s/it]  

[INFO] Procesado frame 50/442


...

[INFO] Procesado frame 350/442


propagate in video:  89%|████████▉ | 391/438 [48:08<08:22, 10.69s/it]

[INFO] Procesado frame 400/442


propagate in video: 100%|██████████| 438/438 [56:33<00:00,  7.75s/it]


[OK] CSV guardado: ./outputs\ball_sam2_track_h2d.csv
[OK] Columnas: ['frame', 'x', 'y', 'xg', 'yg', 'area', 'diam']


In [12]:
# ============================================================
# CHUNK 2/2 — HOMOGRAFÍA + REPRODUCCIÓN (SIN INFERENCIA)
# ------------------------------------------------------------
# Lee el CSV generado por el CHUNK 1:
#   ./outputs/ball_sam2_track_h2d.csv
# Pide clics para homografía VIDEO->MAPA y reproduce minimapa.
# ============================================================

import cv2
import numpy as np
import pandas as pd
import os

# -----------------------------
# CONFIG (AJUSTA ESTO)
# -----------------------------
VIDEO_PATH = r"VideosAnalisis\clip 3 ‐ Hecho con Clipchamp.mp4"
MAP_PATH   = r"beachvolleyballcourt.png"
CSV_PATH   = r"./outputs/ball_sam2_track_h2d.csv"

N_POINTS = 6  # >=4

TARGET_VIDEO_WIDTH = 900
MINIMAP_HEIGHT_RATIO = 0.55

AIR_T0 = 0.20
AIR_T1 = 0.55
SMOOTH_ALPHA_MAP = 0.35


# -----------------------------
# UI homografía (clics)
# -----------------------------
def get_screen_size():
    screen = np.zeros((1, 1, 3), dtype=np.uint8)
    cv2.namedWindow("_tmp", cv2.WINDOW_NORMAL)
    cv2.imshow("_tmp", screen)
    cv2.waitKey(1)
    _, _, w, h = cv2.getWindowImageRect("_tmp")
    cv2.destroyWindow("_tmp")
    return w, h

def resize_to_fit(img, max_w, max_h):
    h, w = img.shape[:2]
    scale = min(max_w / w, max_h / h, 1.0)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    if scale < 1.0:
        out = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    else:
        out = img.copy()
    return out, (w / new_w), (h / new_h)

def _mouse_cb_collect_points(event, x, y, flags, params):
    if event != cv2.EVENT_LBUTTONDOWN:
        return
    pts = params["points"]
    if len(pts) >= params["max_points"]:
        return
    img = params["image"]
    wname = params["wname"]

    pts.append([x, y])
    cv2.circle(img, (x, y), 7, (0, 0, 255), -1)
    cv2.putText(img, f"{len(pts)}", (x + 8, y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.imshow(wname, img)

    if len(pts) == params["max_points"]:
        cv2.waitKey(250)
        cv2.destroyWindow(wname)

def collect_correspondences(imgA, imgB, n_points=6, titleA="VIDEO", titleB="MAPA"):
    sw, sh = get_screen_size()
    max_w = int(sw * 0.85)
    max_h = int(sh * 0.85)

    # A
    dispA, sxA, syA = resize_to_fit(imgA, max_w, max_h)
    ptsA_disp = []
    imgA_draw = dispA.copy()
    wA = f"{titleA}: clica {n_points} puntos (ESC cancela)"
    cv2.namedWindow(wA, cv2.WINDOW_NORMAL)
    cv2.imshow(wA, imgA_draw)
    cv2.setMouseCallback(wA, _mouse_cb_collect_points,
                         {"points": ptsA_disp, "image": imgA_draw, "wname": wA, "max_points": n_points})
    while True:
        k = cv2.waitKey(20) & 0xFF
        if k == 27:
            cv2.destroyAllWindows()
            raise RuntimeError("Cancelado puntos VIDEO.")
        if len(ptsA_disp) >= n_points:
            break
    ptsA = np.array([[p[0] * sxA, p[1] * syA] for p in ptsA_disp], dtype=np.float32)

    # B
    dispB, sxB, syB = resize_to_fit(imgB, max_w, max_h)
    ptsB_disp = []
    imgB_draw = dispB.copy()
    wB = f"{titleB}: clica {n_points} puntos correspondientes (ESC cancela)"
    cv2.namedWindow(wB, cv2.WINDOW_NORMAL)
    cv2.imshow(wB, imgB_draw)
    cv2.setMouseCallback(wB, _mouse_cb_collect_points,
                         {"points": ptsB_disp, "image": imgB_draw, "wname": wB, "max_points": n_points})
    while True:
        k = cv2.waitKey(20) & 0xFF
        if k == 27:
            cv2.destroyAllWindows()
            raise RuntimeError("Cancelado puntos MAPA.")
        if len(ptsB_disp) >= n_points:
            break
    ptsB = np.array([[p[0] * sxB, p[1] * syB] for p in ptsB_disp], dtype=np.float32)

    cv2.destroyAllWindows()
    return ptsA, ptsB


# -----------------------------
# Mezcla adaptativa
# -----------------------------
def clamp01(x):
    return float(max(0.0, min(1.0, x)))

def mix2(a, b, w):
    return (1.0 - w) * a + w * b

def air_weight(cy, yg, diam):
    if not np.isfinite(cy) or not np.isfinite(yg) or not np.isfinite(diam) or diam <= 1e-6:
        return 0.0
    score = (float(yg) - float(cy)) / float(diam)
    w = (score - AIR_T0) / (AIR_T1 - AIR_T0 + 1e-9)
    return clamp01(w)


# -----------------------------
# MAIN
# -----------------------------
def main():
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"No existe CSV: {CSV_PATH}")
    if not os.path.exists(MAP_PATH):
        raise FileNotFoundError(f"No existe mapa: {MAP_PATH}")

    df = pd.read_csv(CSV_PATH)
    needed = {"frame", "x", "y", "xg", "yg", "diam"}
    if not needed.issubset(df.columns):
        raise ValueError(f"CSV no tiene columnas requeridas {sorted(needed)}. Tiene: {list(df.columns)}")

    mapa = cv2.imread(MAP_PATH)
    if mapa is None:
        raise RuntimeError(f"No se pudo leer mapa: {MAP_PATH}")

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir video: {VIDEO_PATH}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    delay = max(1, int(round(1000 / fps))) if fps and fps > 1e-3 else 30

    ret, first_frame = cap.read()
    if not ret:
        cap.release()
        raise RuntimeError("No pude leer primer frame.")
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    print("[INFO] Selecciona puntos para homografía VIDEO -> MAPA")
    pA, pB = collect_correspondences(first_frame, mapa, n_points=N_POINTS, titleA="VIDEO", titleB="MAPA")
    Hm, _ = cv2.findHomography(pA, pB, method=cv2.RANSAC)
    if Hm is None:
        cap.release()
        raise RuntimeError("No se pudo calcular homografía.")

    last_map = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        f_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
        mapa_out = mapa.copy()

        row_df = df[df["frame"] == f_idx]
        if len(row_df) > 0:
            r = row_df.iloc[0]
            if pd.notna(r["x"]) and pd.notna(r["y"]) and pd.notna(r["xg"]) and pd.notna(r["yg"]):
                cx, cy = float(r["x"]), float(r["y"])
                xg, yg = float(r["xg"]), float(r["yg"])
                diam = float(r["diam"]) if pd.notna(r["diam"]) else 1.0

                pt_c = np.array([[cx, cy]], dtype=np.float32).reshape(-1, 1, 2)
                pt_g = np.array([[xg, yg]], dtype=np.float32).reshape(-1, 1, 2)
                xm_c, ym_c = cv2.perspectiveTransform(pt_c, Hm).reshape(-1, 2)[0]
                xm_g, ym_g = cv2.perspectiveTransform(pt_g, Hm).reshape(-1, 2)[0]

                w = air_weight(cy, yg, diam)
                xm = mix2(xm_c, xm_g, w)
                ym = mix2(ym_c, ym_g, w)

                if SMOOTH_ALPHA_MAP > 0:
                    if last_map is None:
                        last_map = (xm, ym)
                    else:
                        xm = SMOOTH_ALPHA_MAP * xm + (1 - SMOOTH_ALPHA_MAP) * last_map[0]
                        ym = SMOOTH_ALPHA_MAP * ym + (1 - SMOOTH_ALPHA_MAP) * last_map[1]
                        last_map = (xm, ym)

                # Debug en vídeo: centroide amarillo, ground cian
                cv2.circle(frame, (int(cx), int(cy)), 6, (0, 255, 255), -1)
                cv2.circle(frame, (int(xg), int(yg)), 6, (255, 255, 0), -1)
                cv2.putText(frame, f"BALL w={w:.2f}", (int(cx) + 10, int(cy) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                cv2.circle(mapa_out, (int(xm), int(ym)), 18, (255, 255, 255), -1)
                cv2.putText(mapa_out, "Pelota", (int(xm) + 10, int(ym) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        fh, fw = frame.shape[:2]
        frame_r = cv2.resize(frame, (TARGET_VIDEO_WIDTH, int(fh * TARGET_VIDEO_WIDTH / fw)))
        target_h = frame_r.shape[0]

        minimap_target_h = int(target_h * MINIMAP_HEIGHT_RATIO)
        mapa_r = cv2.resize(mapa_out, (int(mapa_out.shape[1] * minimap_target_h / mapa_out.shape[0]), minimap_target_h))

        padding_top = (target_h - minimap_target_h) // 2
        padding_bottom = target_h - minimap_target_h - padding_top
        mapa_padded = cv2.copyMakeBorder(mapa_r, padding_top, padding_bottom, 0, 0,
                                         cv2.BORDER_CONSTANT, value=[0, 0, 0])

        combined = np.hstack((frame_r, mapa_padded))
        cv2.imshow("Video + Minimapa (Opcion A: mezcla homografia)", combined)

        if cv2.waitKey(delay) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    print("[OK] Fin.")

if __name__ == "__main__":
    main()


[INFO] Selecciona puntos para homografía VIDEO -> MAPA
[OK] Fin.
